# 1 Exploración inicial de datos


Realizaremos un análisis exploratorio permitirá identificar patrones de compra, relaciones entre productos y posibles limitaciones de los datos que deban considerarse durante el desarrollo.

In [1]:
import pandas as pd

products = pd.read_csv('C:\\Users\\Lucinda\\Downloads\\Prueba tecnica\\Data\\products.csv')
sales = pd.read_csv('C:\\Users\\Lucinda\\Downloads\\Prueba tecnica\\Data\\sales.csv')

In [2]:

print("Dimensiones de productos:", products.shape)
print("Dimensiones de ventas:", sales.shape)

display(products.head())
display(sales.head())


Dimensiones de productos: (28, 8)
Dimensiones de ventas: (89, 5)


,sku,nombre,descripcion,categoria,material,uso_recomendado,precio,stock
0,SKU001,Soplete de gas MAPP,Soplete portátil para soldadura y corte menor ...,herramienta,acero y latón,soldadura y corte menor,450,15
1,SKU002,Varilla de soldar estaño 60/40,Varilla de aporte para soldadura de baja tempe...,consumible,estaño-plomo,soldadura eléctrica y electrónica,80,40
2,SKU003,Varilla de soldar plata,Varilla de aporte para uniones de alta resiste...,consumible,aleación de plata,soldadura de alta resistencia,220,10
3,SKU004,Cartucho de gas MAPP,Cartucho de gas combustible de repuesto para s...,consumible,gas MAPP,combustible para soplete,120,30
4,SKU005,Tornillo acero al carbón 1/4 (caja 100),Tornillería estándar sin recubrimiento,fijación,acero al carbón,interior,60,200


,ticket_id,sku,cantidad,tienda,fecha
0,T001,SKU001,1,Monterrey,2026-02-03
1,T001,SKU004,2,Monterrey,2026-02-03
2,T001,SKU002,3,Monterrey,2026-02-03
3,T001,SKU020,1,Monterrey,2026-02-03
4,T002,SKU001,1,Monterrey,2026-02-06


In [3]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sku              28 non-null     object
 1   nombre           28 non-null     object
 2   descripcion      28 non-null     object
 3   categoria        28 non-null     object
 4   material         28 non-null     object
 5   uso_recomendado  28 non-null     object
 6   precio           28 non-null     int64 
 7   stock            28 non-null     int64 
dtypes: int64(2), object(6)
memory usage: 1.9+ KB


In [4]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 89 entries, 0 to 88
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   ticket_id  89 non-null     object
 1   sku        89 non-null     object
 2   cantidad   89 non-null     int64 
 3   tienda     89 non-null     object
 4   fecha      89 non-null     object
dtypes: int64(1), object(4)
memory usage: 3.6+ KB


# Exploración de valores faltantes

In [5]:
print("Valores nulos en productos:")
print(products.isnull().sum())

print("\nValores nulos en ventas:")
print(sales.isnull().sum())

Valores nulos en productos:
sku                0
nombre             0
descripcion        0
categoria          0
material           0
uso_recomendado    0
precio             0
stock              0
dtype: int64

Valores nulos en ventas:
ticket_id    0
sku          0
cantidad     0
tienda       0
fecha        0
dtype: int64


###  Análisis de valores faltantes

Se realizó una revisión de valores nulos en los archivos `products.csv` y `sales.csv` utilizando la biblioteca Pandas.

**Resultados:**

- `products.csv`: no se identificaron valores nulos en ninguna de sus ocho columnas.
- `sales.csv`: no se identificaron valores nulos en ninguna de sus cinco columnas.



# Duplicados

In [6]:
# Verificar SKU duplicados en el catálogo
print("SKU duplicados en productos:")
print(products['sku'].duplicated().sum())

# Verificar filas completamente duplicadas
print("\nFilas duplicadas en productos:")
print(products.duplicated().sum())

print("\nFilas duplicadas en ventas:")
print(sales.duplicated().sum())

SKU duplicados en productos:
0

Filas duplicadas en productos:
0

Filas duplicadas en ventas:
0


###  Análisis de duplicados

Se verificó la no existencia de registros duplicados en ambos conjuntos de datos mediante Pandas.

**Resultados:**

- SKU duplicados en `products.csv`: 0.
- Filas completamente duplicadas en `products.csv`: 0.
- Filas completamente duplicadas en `sales.csv`: 0.



# Integridad referencial


¿Todos los productos que aparecen en las ventas existen en el catálogo?

In [7]:
# SKU vendidos que no existen en el catálogo
sku_faltantes = set(sales['sku']) - set(products['sku'])

print("SKU vendidos que no están en el catálogo:")
print(sku_faltantes)

# Cantidad de productos distintos vendidos
print("\nProductos distintos vendidos:")
print(sales['sku'].nunique())

# Cantidad total de productos en el catálogo
print("\nProductos en el catálogo:")
print(products['sku'].nunique())

SKU vendidos que no están en el catálogo:
set()

Productos distintos vendidos:
27

Productos en el catálogo:
28


### Validación de integridad referencial

Se verificó que los SKU presentes en el historial de ventas correspondieran a productos existentes en el catálogo.

**Resultados:**

- Productos distintos en el catálogo: 28.
- Productos distintos registrados en ventas: 27.
- SKU vendidos sin correspondencia en el catálogo: 0.
- Productos del catálogo sin ventas registradas: 1.


Existe integridad referencial entre las ventas históricas y el catálogo de productos.

Sin embargo, se identificó un producto sin historial de ventas. Este caso representa una limitación para los sistemas de recomendación basados exclusivamente en interacciones históricas.

Por esta razón, se considerará un enfoque híbrido que combine patrones de compra con características de los productos, permitiendo generar recomendaciones incluso cuando exista poca o ninguna información histórica.

# ¿Cuál es el producto único?

In [8]:
productos_sin_ventas = products[
    ~products['sku'].isin(sales['sku'])
]

display(productos_sin_ventas)

,sku,nombre,descripcion,categoria,material,uso_recomendado,precio,stock
26,SKU027,Regulador de presión para gas MAPP,Regulador de flujo para cartucho de gas combus...,herramienta,latón,control de presión en soldadura,300,8


###  Identificación de productos sin historial de ventas

Se identificó un producto del catálogo que no aparece en el historial de transacciones:

**SKU027 — Regulador de presión para gas MAPP**

- Precio: $300.
- Inventario disponible: 8 unidades.
- Ventas registradas: 0.

**Interpretación:**

No existe información histórica suficiente de este producto para estimar sus asociaciones con otros productos mediante técnicas tradicionales de análisis de transacciones.

**Implicación para el modelo:**

Se propone evaluar un sistema híbrido que incorpore tanto asociaciones históricas como atributos descriptivos de los productos.

Esto permitiría considerar productos sin historial de ventas, siempre que exista una relación de complementariedad justificable con el producto consultado y disponibilidad en inventario.

# Análisis de Tickets

¿Cuántos productos distintos contiene cada ticket?

In [9]:
# Cantidad de productos distintos por ticket
productos_por_ticket = (
    sales.groupby('ticket_id')['sku']
    .nunique()
)

print(productos_por_ticket.describe())

count    42.000000
mean      2.119048
std       0.395239
min       2.000000
25%       2.000000
50%       2.000000
75%       2.000000
max       4.000000
Name: sku, dtype: float64


Análisis de la composición de los tickets
La distribución muestra que las transacciones contienen principalmente dos productos distintos, con una variabilidad relativamente baja.

Esto favorece el análisis de asociaciones entre pares de productos como punto de partida para el sistema de recomendaciones.

Consideraciones metodológicas:

El número reducido de transacciones puede generar estimaciones inestables de las asociaciones. Por ello, será necesario considerar la frecuencia de aparición de cada relación, evaluar su confiabilidad y evitar interpretar asociaciones poco frecuentes como patrones concluyentes.

# 2 Selección de relaciones candidatas

A partir de los atributos del catálogo, se identificarán posibles relaciones de complementariedad entre productos.

La selección inicial se basó en los campos nombre, descripción, material y uso_recomendado.

Estas relaciones constituyen hipótesis técnicas y no deben interpretarse como compatibilidades verificadas.

# Metodología para identificar relaciones entre productos



**Objetivo**

Construir una base de relaciones técnicas entre productos mediante un procedimiento sistemático, reproducible y respaldado por fuentes externas.

**Planteamiento**

El catálogo contiene 28 productos, lo que genera 378 parejas no ordenadas posibles.

La evaluación manual de todas las combinaciones puede resultar ineficiente. Por ello, se propone un procedimiento de selección de candidatos basado en los atributos disponibles.

**Metodología**

1. Generar las combinaciones posibles de productos.
2. Analizar sus atributos descriptivos para identificar relaciones potenciales.
3. Priorizar las parejas que presenten evidencia preliminar de relación funcional.
4. Investigar fuentes técnicas externas para verificar complementariedad y compatibilidad.
5. Clasificar las relaciones como verificadas, descartadas o no determinadas.
6. Construir una matriz de complementariedad documentada.

**Consideraciones metodológicas**

La similitud entre atributos no implica necesariamente complementariedad. Por tanto, las relaciones generadas automáticamente se considerarán hipótesis hasta contar con evidencia técnica suficiente.

Las asociaciones observadas en las ventas históricas se analizarán por separado y no se utilizarán como prueba de compatibilidad técnica.

Las relaciones sin evidencia suficiente conservarán el estado de no determinadas, en lugar de clasificarse automáticamente como incompatibles.

# Paso 1: Generación del espacio de parejas candidatas



Se generarán todas las combinaciones no ordenadas de dos productos distintos del catálogo mediante itertools.combinations.

Para un catálogo de (n) productos, el número de parejas posibles está dado por:

[
\binom{n}{2}=\frac{n(n-1)}{2}
]

Con (n=28), el espacio de búsqueda contiene 378 parejas candidatas.

Esta generación exhaustiva evita seleccionar arbitrariamente relaciones antes de aplicar criterios de análisis.

Nota metodológica: las parejas generadas no representan asociaciones verificadas. Constituyen el espacio inicial sobre el cual se aplicarán métodos de selección y validación técnica.

In [10]:
from itertools import combinations

# Obtener los SKU únicos del catálogo
sku_list = products['sku'].unique()

# Generar todas las parejas posibles
parejas = list(combinations(sku_list, 2))

print("Número de parejas generadas:", len(parejas))
print("Primeras 10 parejas:")
print(parejas[:10])

Número de parejas generadas: 378
Primeras 10 parejas:
[('SKU001', 'SKU002'), ('SKU001', 'SKU003'), ('SKU001', 'SKU004'), ('SKU001', 'SKU005'), ('SKU001', 'SKU006'), ('SKU001', 'SKU007'), ('SKU001', 'SKU008'), ('SKU001', 'SKU009'), ('SKU001', 'SKU010'), ('SKU001', 'SKU011')]


# Paso 2: Representación matemática de los productos

Nuestro catálogo contiene información textual sobre cada producto:
- Nombre
- Descripción
- Categoría
- Material
- Uso recomendado
Podemos transformar esa información en vectores numéricos para comparar los productos matemáticamente.

Se integrarán los atributos descriptivos del catálogo: nombre, descripción, categoría, material y uso recomendado.

Esta representación permitirá aplicar TF-IDF y calcular medidas de similitud entre productos.
Utilizaremos TF-IDF y similitud del coseno.

TF-IDF asigna importancia a las palabras según su frecuencia en cada descripción y su presencia en el conjunto de documentos.


Hipótesis metodológica: los atributos textuales pueden aportar información útil para priorizar relaciones candidatas, aunque la similitud semántica o léxica no constituye evidencia suficiente de complementariedad técnica.

Las relaciones seleccionadas deberán verificarse posteriormente mediante fuentes técnicas externas.




In [11]:
# Crear una descripción integrada de cada producto es decir construir la representación textual.
products['texto'] = (
    products['nombre'].fillna('') + ' ' +
    products['descripcion'].fillna('') + ' ' +
    products['categoria'].fillna('') + ' ' +
    products['material'].fillna('') + ' ' +
    products['uso_recomendado'].fillna('')
)

display(products[['sku', 'texto']].head())

,sku,texto
0,SKU001,Soplete de gas MAPP Soplete portátil para sold...
1,SKU002,Varilla de soldar estaño 60/40 Varilla de apor...
2,SKU003,Varilla de soldar plata Varilla de aporte para...
3,SKU004,Cartucho de gas MAPP Cartucho de gas combustib...
4,SKU005,Tornillo acero al carbón 1/4 (caja 100) Tornil...


# Paso 3: Vectorización mediante TF-IDF


Se propone utilizar TF-IDF (*Term Frequency–Inverse Document Frequency*) para transformar los atributos textuales de los productos en representaciones numéricas.

La técnica pondera los términos según su frecuencia en cada documento y su presencia en el conjunto de documentos.

Se utilizarán unigramas y bigramas, con normalización de mayúsculas y acentos.

La matriz resultante tendrá 28 filas, correspondientes a los productos, y un número de columnas determinado por el vocabulario identificado.

**Limitación:** TF-IDF mide coincidencias léxicas ponderadas, no compatibilidad funcional. Por ello, sus resultados se utilizarán para priorizar relaciones candidatas, sin descartar automáticamente aquellas con baja similitud.


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents='unicode',
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(products['texto'])

print("Dimensiones de la matriz TF-IDF:")
print(tfidf_matrix.shape)

Dimensiones de la matriz TF-IDF:
(28, 515)


# Paso 4: Cálculo de similitud mediante coseno

La representación TF-IDF generó una matriz de dimensiones (28 \times 515), donde cada producto está descrito mediante 515 características textuales.

Se utiliza la similitud del coseno para cuantificar la proximidad entre las representaciones vectoriales de los productos.

Para dos vectores (x_i) y (x_j):

[
\operatorname{sim}(i,j)=
\frac{x_i\cdot x_j}{|x_i||x_j|}
]

La matriz de similitud resultante tendrá dimensiones (28 \times 28).

Interpretación metodológica:

La similitud del coseno permitirá priorizar parejas candidatas para su posterior análisis técnico.

Los valores obtenidos no se interpretarán como probabilidades de compra ni como evidencia directa de complementariedad.

Las relaciones deberán contrastarse con información técnica externa antes de incorporarse como reglas verificadas al sistema de recomendaciones.

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

# Calcular la similitud entre todos los productos
similarity_matrix = cosine_similarity(tfidf_matrix)

# Convertir la matriz en un DataFrame
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=products['sku'],
    columns=products['sku']
)

print("Dimensiones:", similarity_df.shape)

display(similarity_df.iloc[:5, :5].round(3))

Dimensiones: (28, 28)


sku,SKU001,SKU002,SKU003,SKU004,SKU005
sku,,,,,
SKU001,1.000,0.080,0.047,0.379,0.017
SKU002,0.080,1.000,0.331,0.042,0.000
SKU003,0.047,0.331,1.000,0.052,0.000
SKU004,0.379,0.042,0.052,1.000,0.000
SKU005,0.017,0.000,0.000,0.000,1.000


#  Interpretación de la matriz de similitud

Se calculó la similitud del coseno entre los vectores TF-IDF de los 28 productos, obteniendo una matriz simétrica de dimensiones \(28 \times 28\).

**Propiedades matemáticas:**

- La matriz es simétrica debido a la conmutatividad del producto interno.
- La diagonal principal contiene valores iguales a 1, ya que cada vector se compara consigo mismo.
- Los coeficientes se encuentran entre 0 y 1 debido a la representación TF-IDF no negativa.

**Resultados preliminares:**

Entre las primeras cinco referencias analizadas, destacan las parejas SKU001–SKU004, con similitud de 0.379, y SKU002–SKU003, con similitud de 0.331.

**Interpretación:**

Estos valores indican coincidencias relativamente mayores entre sus representaciones textuales, pero no permiten establecer directamente relaciones de complementariedad.

La similitud textual se utilizará como herramienta de priorización para la investigación técnica posterior.

**Limitación:**

El método puede asignar similitudes elevadas a productos sustitutos y similitudes bajas a productos funcionalmente complementarios. Por ello, será necesario contrastar los resultados con conocimiento técnico externo.

# Paso 6: Ordenar las 378 parejas.


In [14]:
# Asociar cada pareja con su similitud
resultados = []

for sku1, sku2 in parejas:
    similitud = similarity_df.loc[sku1, sku2]

    resultados.append({
        'sku_origen': sku1,
        'sku_destino': sku2,
        'similitud': similitud
    })

# Crear un DataFrame y ordenar
relaciones_df = pd.DataFrame(resultados)

relaciones_df = relaciones_df.sort_values(
    by='similitud',
    ascending=False
).reset_index(drop=True)

display(relaciones_df.head(15))

,sku_origen,sku_destino,similitud
0,SKU022,SKU023,0.575972
1,SKU005,SKU009,0.559564
2,SKU005,SKU024,0.541719
3,SKU009,SKU024,0.499026
4,SKU008,SKU009,0.451493
5,SKU011,SKU016,0.438147
6,SKU007,SKU025,0.394554
7,SKU006,SKU008,0.389454
8,SKU001,SKU004,0.378771
9,SKU004,SKU027,0.366162


# Paso 7: Agregar información del catálogo a las parejas.



Se generaron las 378 combinaciones posibles de dos productos diferentes y se asignó a cada pareja su coeficiente de similitud del coseno.

Las parejas se ordenaron de mayor a menor similitud y se enriquecieron con los nombres de los productos para facilitar su interpretación.

La mayor similitud observada corresponde a SKU022–SKU023, con un coeficiente aproximado de 0.576.

**Criterio metodológico:** el ordenamiento constituye una priorización exploratoria, no una clasificación definitiva de complementariedad. No se establece todavía un umbral de aceptación, ya que sería necesario justificarlo y evaluar sus consecuencias.

En las siguientes etapas se contrastarán las relaciones candidatas con el historial de compras y con fuentes técnicas externas, manteniendo separadas la evidencia estadística y la evidencia de compatibilidad.

In [15]:
# Crear un diccionario SKU -> nombre del producto
nombres = products.set_index('sku')['nombre']

# Incorporar los nombres a la tabla de relaciones
relaciones_df['producto_origen'] = relaciones_df['sku_origen'].map(nombres)
relaciones_df['producto_destino'] = relaciones_df['sku_destino'].map(nombres)

# Mostrar las primeras 15 parejas
display(
    relaciones_df[
        ['sku_origen', 'producto_origen',
         'sku_destino', 'producto_destino',
         'similitud']
    ].head(15)
)

,sku_origen,producto_origen,sku_destino,producto_destino,similitud
0,SKU022,Cable eléctrico calibre 12 rollo 100m,SKU023,Cable eléctrico uso rudo rollo 100m,0.575972
1,SKU005,Tornillo acero al carbón 1/4 (caja 100),SKU009,Lámina acero al carbón calibre 24,0.559564
2,SKU005,Tornillo acero al carbón 1/4 (caja 100),SKU024,Candado acero al carbón,0.541719
3,SKU009,Lámina acero al carbón calibre 24,SKU024,Candado acero al carbón,0.499026
4,SKU008,Lámina galvanizada calibre 24,SKU009,Lámina acero al carbón calibre 24,0.451493
5,SKU011,Tubo CPVC estabilizado UV 3/4,SKU016,Pintura esmalte resistente UV,0.438147
6,SKU007,Tornillo acero inoxidable 316 1/4 (caja 100),SKU025,Candado acero inoxidable marino,0.394554
7,SKU006,Tornillo acero galvanizado 1/4 (caja 100),SKU008,Lámina galvanizada calibre 24,0.389454
8,SKU001,Soplete de gas MAPP,SKU004,Cartucho de gas MAPP,0.378771
9,SKU004,Cartucho de gas MAPP,SKU027,Regulador de presión para gas MAPP,0.366162


# Paso 8: Análisis de coocurrencia en tickets de compra

El análisis de similitud textual mostró que las coincidencias en materiales, características o descripciones no implican necesariamente complementariedad funcional.

Por ello, se incorpora una segunda fuente de evidencia: el historial de ventas.

Se agrupan los productos por identificador de ticket para estudiar qué referencias fueron adquiridas conjuntamente.

Cada ticket se representa mediante el conjunto de SKU distintos que contiene, evitando contabilizar múltiples unidades de un mismo producto como apariciones independientes.

Este procedimiento permitirá calcular frecuencias de coocurrencia y, posteriormente, métricas de asociación como soporte, confianza y *lift*.

**Limitación:** la compra conjunta constituye evidencia de asociación comercial, pero no demuestra por sí sola compatibilidad técnica ni causalidad.

In [16]:
# Agrupar los SKU comprados en cada ticket
tickets = (
    sales.groupby('ticket_id')['sku']
    .apply(lambda x: sorted(set(x)))
    .reset_index(name='productos')
)

display(tickets.head(10))

print("Número de tickets:", len(tickets))

,ticket_id,productos
0,T001,"[SKU001, SKU002, SKU004, SKU020]"
1,T002,"[SKU001, SKU004, SKU026]"
2,T003,"[SKU017, SKU019, SKU021]"
3,T004,"[SKU017, SKU022]"
4,T005,"[SKU003, SKU020, SKU021]"
5,T006,"[SKU002, SKU004]"
6,T007,"[SKU009, SKU019]"
7,T008,"[SKU020, SKU021]"
8,T009,"[SKU001, SKU003]"
9,T010,"[SKU017, SKU018]"


Número de tickets: 42


# Paso 9: Calcular la frecuencia de coocurrencia.
A partir de los 42 tickets históricos, se identifican todas las parejas de productos distintos adquiridos en una misma transacción.

Para cada pareja ((A,B)), se calcula:

[
N(A,B)=\sum_{t=1}^{42}\mathbb{1}{A\in t \land B\in t}
]

Esta métrica representa el número de tickets en los que ambos productos aparecen conjuntamente.

Se utiliza itertools.combinations para generar parejas sin considerar el orden, y collections.Counter para acumular sus frecuencias.

Las parejas con frecuencia cero no se interpretan como incompatibles, sino como relaciones sin evidencia de coocurrencia en la muestra disponible.

Dado el tamaño reducido del historial, estas frecuencias se consideran señales exploratorias y no conclusiones definitivas sobre los hábitos de compra.

In [17]:
from collections import Counter
from itertools import combinations

# Contar las parejas presentes en los tickets
conteo_parejas = Counter()

for productos_ticket in tickets['productos']:
    for pareja in combinations(productos_ticket, 2):
        conteo_parejas[pareja] += 1

# Convertir los conteos en una tabla
coocurrencias_df = pd.DataFrame(
    [
        {
            'sku_origen': sku1,
            'sku_destino': sku2,
            'tickets_juntos': frecuencia
        }
        for (sku1, sku2), frecuencia in conteo_parejas.items()
    ]
)

coocurrencias_df = coocurrencias_df.sort_values(
    'tickets_juntos',
    ascending=False
).reset_index(drop=True)

display(coocurrencias_df.head(15))

,sku_origen,sku_destino,tickets_juntos
0,SKU011,SKU013,2
1,SKU020,SKU021,2
2,SKU001,SKU004,2
3,SKU002,SKU004,2
4,SKU011,SKU016,2
5,SKU010,SKU012,2
6,SKU005,SKU024,2
7,SKU017,SKU018,2
8,SKU014,SKU023,1
9,SKU006,SKU008,1


# Paso 10: Métricas de asociación entre productos. Calcular soporte, confianza y lift.


Se calcularon tres métricas para evaluar las relaciones de compra conjunta: soporte, confianza y lift.

El soporte mide la proporción de tickets que contienen ambos productos.

La confianza estima la frecuencia relativa con la que se observa un producto B entre los tickets que contienen A. Es una métrica direccional.

El lift compara la frecuencia conjunta observada con la esperada si las apariciones de ambos productos fueran independientes.

Estas métricas permiten distinguir entre parejas que aparecen juntas por su popularidad individual y parejas con una asociación relativamente elevada.

Limitación estadística: el historial contiene solamente 42 tickets y ninguna pareja presenta más de dos coocurrencias. Por lo tanto, las métricas pueden ser inestables, especialmente para productos con baja frecuencia individual. Los resultados se utilizarán como señales exploratorias, no como evidencia concluyente de complementariedad.

In [18]:
# Número total de tickets
n_tickets = len(tickets)

# Contar en cuántos tickets aparece cada producto
frecuencia_productos = Counter()

for productos_ticket in tickets['productos']:
    frecuencia_productos.update(productos_ticket)

# Calcular métricas de asociación
metricas = []

for (sku1, sku2), juntos in conteo_parejas.items():
    n_a = frecuencia_productos[sku1]
    n_b = frecuencia_productos[sku2]

    soporte = juntos / n_tickets
    confianza_ab = juntos / n_a
    confianza_ba = juntos / n_b

    lift = soporte / (
        (n_a / n_tickets) * (n_b / n_tickets)
    )

    metricas.append({
        'sku_origen': sku1,
        'sku_destino': sku2,
        'tickets_juntos': juntos,
        'soporte': soporte,
        'confianza_ab': confianza_ab,
        'confianza_ba': confianza_ba,
        'lift': lift
    })

metricas_df = pd.DataFrame(metricas)

display(
    metricas_df.sort_values(
        'lift', ascending=False
    ).head(15).round(3)
)

,sku_origen,sku_destino,tickets_juntos,soporte,confianza_ab,confianza_ba,lift
23,SKU023,SKU028,1,0.024,0.333,1.000,14.000
3,SKU002,SKU004,2,0.048,1.000,0.667,14.000
6,SKU001,SKU026,1,0.024,0.333,1.000,14.000
7,SKU004,SKU026,1,0.024,0.333,1.000,14.000
1,SKU001,SKU004,2,0.048,0.667,0.667,9.333
17,SKU017,SKU018,2,0.048,0.400,1.000,8.400
12,SKU003,SKU020,1,0.024,0.500,0.333,7.000
16,SKU001,SKU003,1,0.024,0.333,0.500,7.000
14,SKU020,SKU021,2,0.048,0.667,0.500,7.000
0,SKU001,SKU002,1,0.024,0.333,0.500,7.000


# Conclusiones del análisis exploratorio

Se analizaron 28 productos y 42 tickets históricos para identificar posibles relaciones comerciales y técnicas entre los artículos del catálogo.

Se construyeron 378 parejas candidatas y se calcularon sus similitudes textuales mediante TF-IDF y similitud del coseno.

Posteriormente, se estudiaron las compras conjuntas utilizando soporte, confianza y lift.

**Principales hallazgos:**

- La similitud textual no equivale necesariamente a complementariedad funcional.
- Ninguna pareja aparece conjuntamente en más de dos tickets.
- Algunas asociaciones presentan valores elevados de lift, pero están respaldadas por muy pocas observaciones.
- La confianza es direccional, mientras que el lift es simétrico.
- Las relaciones de compatibilidad requieren validación técnica adicional.

**Decisión de diseño:**

Se propone desarrollar un sistema de recomendaciones híbrido y explicable que utilice las señales de similitud textual y asociación comercial para priorizar candidatos.

Las relaciones técnicas verificadas mediante fuentes externas se incorporarán como una fuente independiente de evidencia. Las relaciones no verificadas no se presentarán como compatibilidades confirmadas.

Las recomendaciones deberán considerar la disponibilidad del inventario y ofrecer explicaciones comprensibles para el usuario.

Dado el tamaño reducido de la muestra, los resultados estadísticos se tratarán como evidencia exploratoria.


# 

#  4  Desarrollo del modelo de recomendaciones

# Formulación del problema de decisión

El objetivo es construir un sistema de recomendaciones híbrido, explicable y orientado a la venta cruzada.

Cuando un cliente abra su carrito, el sistema deberá seleccionar hasta tres productos adicionales que puedan complementar su compra.

Sea:

(P): conjunto de productos disponibles en el catálogo.

(B \subseteq P): conjunto de productos presentes en el carrito.

(R \subseteq P \setminus B): conjunto de productos recomendados.

(U(R \mid B)): función de utilidad de recomendar el conjunto (R), dado el carrito (B).

El problema general se expresa como:

[
R^* = \underset{R}{\arg\max}; U(R \mid B)
]

sujeto a:

[
|R| \leq 3
]

y a restricciones adicionales de inventario, elegibilidad y compatibilidad técnica cuando exista evidencia verificable.

La función de utilidad deberá construirse a partir de las señales disponibles y sus limitaciones estadísticas.

Distinción metodológica:

El sistema separará dos problemas:

Estimación: cuantificar la evidencia de relación entre productos.

Optimización: seleccionar el conjunto de recomendaciones que maximice la utilidad definida, respetando las restricciones.

La función de utilidad no se interpretará automáticamente como una probabilidad de compra ni como un incremento causal de ventas.

In [26]:
print("Productos:", products.shape)
print("Ventas:", sales.shape)
print("Tickets:", len(tickets))
print("Parejas:", len(relaciones_df))
print("Métricas:", metricas_df.shape)

Productos: (28, 9)
Ventas: (89, 5)
Tickets: 42
Parejas: 378
Métricas: (45, 7)


# Frecuencias de aparición de productos

###  Frecuencias de aparición de productos

Para estimar las probabilidades condicionales de coocurrencia, se calcula la frecuencia de aparición de cada producto en los tickets históricos.

Sea \(n_i\) el número de tickets distintos que contienen el producto \(i\).

Se considera la presencia o ausencia de cada producto por ticket, independientemente de la cantidad de unidades adquiridas.

Esta definición permite modelar la coocurrencia como un resultado binario y proporciona el denominador necesario para calcular las probabilidades condicionales de asociación.

Las frecuencias obtenidas se utilizarán posteriormente para actualizar una distribución previa Beta y obtener estimaciones suavizadas de asociación entre productos.

In [29]:
from collections import Counter

frecuencia_productos = Counter()

for productos_ticket in tickets["productos"]:
    frecuencia_productos.update(productos_ticket)

frecuencias_df = (
    pd.DataFrame(
        frecuencia_productos.items(),
        columns=["sku", "n_i"]
    )
    .sort_values("n_i", ascending=False)
    .reset_index(drop=True)
)

display(frecuencias_df)

,sku,n_i
0,SKU005,5
1,SKU017,5
2,SKU016,5
3,SKU011,5
4,SKU008,4
5,SKU015,4
6,SKU021,4
7,SKU022,4
8,SKU012,4
9,SKU013,4


In [30]:
print("Suma de frecuencias:", frecuencias_df["n_i"].sum())

print(
    "Total de productos en tickets:",
    tickets["productos"].apply(len).sum()
)

Suma de frecuencias: 89
Total de productos en tickets: 89


# Calcular la estimación bayesiana

e analizó la asociación dirigida entre SKU001 y SKU004.

Se identificaron tres tickets que contienen SKU001, de los cuales dos también contienen SKU004.

La confianza empírica es:

[
\widehat{P}(SKU004\mid SKU001)=\frac{2}{3}\approx0.6667
]

Se utilizó provisionalmente una distribución previa Beta(1,1).

Después de observar dos éxitos y un fracaso, la distribución posterior es Beta(3,2), cuya media es:

[
\mathbb{E}[p\mid D]=\frac{3}{5}=0.60
]

La estimación posterior es menor que la confianza empírica debido al efecto de regularización de la distribución previa.

Dado que únicamente existen tres observaciones del producto de origen, la incertidumbre sigue siendo considerable.

Esta estimación describe una asociación de coocurrencia bajo el modelo estadístico adoptado. No demuestra complementariedad técnica ni representa una probabilidad causal de compra inducida por una recomendación.

La distribución previa utilizada es provisional y deberá evaluarse antes de establecer el modelo definitivo.

In [31]:
sku_i = "SKU001"
sku_j = "SKU004"

n_i = frecuencia_productos[sku_i]

n_ij = sum(
    sku_i in productos_ticket and sku_j in productos_ticket
    for productos_ticket in tickets["productos"]
)

confianza_empirica = n_ij / n_i

alpha = 1
beta = 1

confianza_bayesiana = (n_ij + alpha) / (n_i + alpha + beta)

print("Producto de origen:", sku_i)
print("Producto recomendado:", sku_j)
print("Tickets con producto origen:", n_i)
print("Tickets con ambos productos:", n_ij)
print("Confianza empírica:", round(confianza_empirica, 4))
print("Media posterior bayesiana:", round(confianza_bayesiana, 4))

Producto de origen: SKU001
Producto recomendado: SKU004
Tickets con producto origen: 3
Tickets con ambos productos: 2
Confianza empírica: 0.6667
Media posterior bayesiana: 0.6


In [32]:
n_j = frecuencia_productos["SKU004"]

confianza_inversa = n_ij / n_j

bayes_inversa = (n_ij + 1) / (n_j + 2)

print("Tickets con SKU004:", n_j)
print("Confianza empírica inversa:", round(confianza_inversa, 4))
print("Media posterior inversa:", round(bayes_inversa, 4))

Tickets con SKU004: 3
Confianza empírica inversa: 0.6667
Media posterior inversa: 0.6


#  Construcción de relaciones dirigidas

Se consideran todas las relaciones dirigidas posibles entre los 28 productos del catálogo, excluyendo las relaciones de un producto consigo mismo.

El número total de relaciones es:

\[
28(28-1)=756
\]

Para cada relación \(i\rightarrow j\), necesitamos calcular:

- \(n_i\): número de tickets que contienen el producto de origen.
- \(n_{ij}\): número de tickets que contienen ambos productos.
- La confianza empírica:

\[
\widehat{P}(j\mid i)=\frac{n_{ij}}{n_i}
\]

Esta confianza es direccional: en general, recomendar \(j\) a partir de \(i\) no tiene la misma confianza que recomendar \(i\) a partir de \(j\).

Cuando el producto de origen no aparece en ningún ticket, la confianza empírica queda indefinida.

En esta etapa se construyen las relaciones y sus estadísticas descriptivas. Posteriormente se incorporará el suavizado bayesiano para reducir la influencia de las muestras pequeñas.

In [39]:
from itertools import permutations

resultados = []

skus = products["sku"].tolist()

for sku_i, sku_j in permutations(skus, 2):

    # Número de tickets que contienen el producto de origen
    n_i = frecuencia_productos[sku_i]

    # Número de tickets que contienen ambos productos
    n_ij = sum(
        sku_i in productos_ticket and sku_j in productos_ticket
        for productos_ticket in tickets["productos"]
    )

    # Confianza empírica
    confianza = n_ij / n_i if n_i > 0 else None

    resultados.append({
        "sku_origen": sku_i,
        "sku_recomendado": sku_j,
        "n_i": n_i,
        "n_ij": n_ij,
        "confianza_empirica": confianza
    })

asociaciones_df = pd.DataFrame(resultados)

print("Relaciones dirigidas:", len(asociaciones_df))

display(asociaciones_df.head(10))

Relaciones dirigidas: 756


,sku_origen,sku_recomendado,n_i,n_ij,confianza_empirica
0,SKU001,SKU002,3,1,0.333333
1,SKU001,SKU003,3,1,0.333333
2,SKU001,SKU004,3,2,0.666667
3,SKU001,SKU005,3,0,0.000000
4,SKU001,SKU006,3,0,0.000000
5,SKU001,SKU007,3,0,0.000000
6,SKU001,SKU008,3,0,0.000000
7,SKU001,SKU009,3,0,0.000000
8,SKU001,SKU010,3,0,0.000000
9,SKU001,SKU011,3,0,0.000000


#  Limitaciones de la confianza empírica

La confianza empírica permite medir la frecuencia con la que un producto aparece junto con otro, condicionada a la presencia del producto de origen.

Sin embargo, nuestro conjunto de datos contiene únicamente 42 tickets históricos, por lo que algunas relaciones se estiman utilizando muy pocas observaciones.

Por ejemplo, si dos productos aparecen juntos en el único ticket que contiene al producto de origen, su confianza empírica es del 100 %. Esto no implica que la probabilidad real de compra conjunta sea del 100 %.

Para reducir la influencia de estas muestras pequeñas, se propone utilizar un modelo beta-binomial que incorpore una distribución previa.

Se examinó inicialmente la distribución Beta(1,1), cuya media es del 50 %. Sin embargo, esta distribución no parece apropiada como elección definitiva para nuestro catálogo, ya que las frecuencias de compra de los productos son considerablemente menores.

Por ello, se construirá una distribución previa específica para cada producto recomendado, utilizando su popularidad histórica suavizada.

# Popularidad histórica y suavizado

Para construir una distribución previa más adecuada, utilizamos la frecuencia histórica de cada producto.

Sea:

- \(N=42\): número total de tickets.
- \(n_j\): número de tickets que contienen el producto \(j\).
- \(q_j\): popularidad histórica suavizada del producto \(j\).

Definimos:

\[
q_j=\frac{n_j+\varepsilon}{N+2\varepsilon}
\]

donde \(\varepsilon=0.5\) es un parámetro de suavizado elegido provisionalmente.

Este procedimiento evita asignar una probabilidad exactamente igual a cero a los productos que no aparecen en las ventas históricas.

Por ejemplo, SKU027 no aparece en ningún ticket:

\[
n_{027}=0
\]

Sin suavizado, su frecuencia relativa sería cero. Con el suavizado propuesto:

\[
q_{027}=\frac{0+0.5}{42+1}\approx0.01163
\]

Así obtenemos una probabilidad base positiva, aunque pequeña.

Esta popularidad suavizada se utilizará para definir las distribuciones previas del modelo bayesiano. No representa por sí misma una probabilidad de compra inducida por una recomendación.

#  Construcción de las distribuciones previas

Para cada relación dirigida \(i\rightarrow j\), consideramos una probabilidad condicional desconocida \(p_{j\mid i}\).

Suponemos que esta probabilidad tiene una distribución previa Beta:

\[
p_{j\mid i}\sim\operatorname{Beta}(\alpha_j,\beta_j)
\]

Los parámetros de la distribución se definen mediante:

\[
\alpha_j=kq_j
\]

\[
\beta_j=k(1-q_j)
\]

Donde:

- \(q_j\) es la popularidad suavizada del producto recomendado.
- \(k=5\) es la intensidad de regularización elegida provisionalmente.
- \(\alpha_j\) y \(\beta_j\) son los parámetros de la distribución Beta.

La media previa es:

\[
\mathbb{E}[p_{j\mid i}]
=\frac{\alpha_j}{\alpha_j+\beta_j}
=q_j
\]

Esto significa que, antes de considerar las compras conjuntas de una pareja específica, utilizamos la popularidad histórica del producto recomendado como punto de partida.

El parámetro \(k\) controla cuánto influye esa información previa en la estimación posterior. Su valor deberá evaluarse posteriormente, ya que no ha sido optimizado con los datos disponibles.

La distribución previa describe un supuesto estadístico sobre la coocurrencia de productos; no establece compatibilidad técnica ni complementariedad comercial.

In [41]:
# 4.2.6 y 4.2.7
# Popularidad suavizada y parámetros de la distribución Beta

N = len(tickets)
epsilon = 0.5
k = 5

# Frecuencia histórica de cada producto
popularidad_df = products[["sku", "nombre"]].copy()

popularidad_df["n_j"] = (
    popularidad_df["sku"]
    .map(frecuencia_productos)
    .fillna(0)
    .astype(int)
)

# Popularidad suavizada
popularidad_df["q_j"] = (
    (popularidad_df["n_j"] + epsilon)
    / (N + 2 * epsilon)
)

# Parámetros de la distribución previa Beta
popularidad_df["alpha_j"] = k * popularidad_df["q_j"]

popularidad_df["beta_j"] = k * (
    1 - popularidad_df["q_j"]
)

# Validaciones
assert len(popularidad_df) == len(products)
assert (popularidad_df["alpha_j"] > 0).all()
assert (popularidad_df["beta_j"] > 0).all()

print("Total de tickets:", N)
print("Total de productos:", len(popularidad_df))
print("Parámetros válidos: True")

display(popularidad_df)

Total de tickets: 42
Total de productos: 28
Parámetros válidos: True


,sku,nombre,n_j,q_j,alpha_j,beta_j
0,SKU001,Soplete de gas MAPP,3,0.081395,0.406977,4.593023
1,SKU002,Varilla de soldar estaño 60/40,2,0.058140,0.290698,4.709302
2,SKU003,Varilla de soldar plata,2,0.058140,0.290698,4.709302
3,SKU004,Cartucho de gas MAPP,3,0.081395,0.406977,4.593023
4,SKU005,Tornillo acero al carbón 1/4 (caja 100),5,0.127907,0.639535,4.360465
5,SKU006,Tornillo acero galvanizado 1/4 (caja 100),4,0.104651,0.523256,4.476744
6,SKU007,Tornillo acero inoxidable 316 1/4 (caja 100),4,0.104651,0.523256,4.476744
7,SKU008,Lámina galvanizada calibre 24,4,0.104651,0.523256,4.476744
8,SKU009,Lámina acero al carbón calibre 24,3,0.081395,0.406977,4.593023
9,SKU010,Tubo PVC cédula 40 3/4,3,0.081395,0.406977,4.593023


# 4.2.8 Actualización bayesiana de las relaciones dirigidas

Una vez construidas las relaciones dirigidas y definidas las distribuciones previas, podemos calcular la distribución posterior de cada probabilidad condicional.

Para una relación \(i\rightarrow j\), consideramos:

- \(n_i\): número de tickets que contienen el producto de origen.
- \(n_{ij}\): número de tickets que contienen ambos productos.
- \(\alpha_j\) y \(\beta_j\): parámetros de la distribución previa del producto recomendado.

Suponiendo un modelo binomial para las coocurrencias, la distribución posterior es:

\[
p_{j\mid i}\mid D
\sim
\operatorname{Beta}
(\alpha_j+n_{ij},\ \beta_j+n_i-n_{ij})
\]

La media posterior es:

\[
\widehat p_{j\mid i}
=
\frac{n_{ij}+\alpha_j}
{n_i+\alpha_j+\beta_j}
\]

Como \(\alpha_j+\beta_j=k\), podemos escribir:

\[
\widehat p_{j\mid i}
=
\frac{n_{ij}+kq_j}{n_i+k}
\]

Esta estimación combina la evidencia de compras conjuntas con la popularidad histórica suavizada del producto recomendado.

Cuando existen pocas observaciones, la distribución previa tiene mayor influencia. A medida que aumenta \(n_i\), la información observada adquiere mayor peso.

La media posterior es una estimación de asociación condicional bajo los supuestos del modelo. No demuestra compatibilidad técnica ni permite afirmar que una recomendación incrementará causalmente las ventas.

Los parámetros \(k=5\) y \(\varepsilon=0.5\) son provisionales y deberán evaluarse antes de considerar definitivo el modelo.

In [42]:
#  Actualización bayesiana

# Recuperamos los parámetros de cada producto recomendado
parametros = popularidad_df.set_index("sku")

# Agregamos la popularidad suavizada
asociaciones_df["q_j"] = (
    asociaciones_df["sku_recomendado"]
    .map(parametros["q_j"])
)

# Agregamos el parámetro alpha
asociaciones_df["alpha_j"] = (
    asociaciones_df["sku_recomendado"]
    .map(parametros["alpha_j"])
)

# Calculamos la media posterior
asociaciones_df["media_posterior"] = (
    asociaciones_df["n_ij"] + asociaciones_df["alpha_j"]
) / (
    asociaciones_df["n_i"] + k
)

# Comprobamos que no existan valores faltantes
assert asociaciones_df["media_posterior"].notna().all()

print("Relaciones evaluadas:", len(asociaciones_df))

display(
    asociaciones_df.sort_values(
        "media_posterior",
        ascending=False
    ).head(10)
)

Relaciones evaluadas: 756


,sku_origen,sku_recomendado,n_i,n_ij,confianza_empirica,q_j,alpha_j,media_posterior
475,SKU018,SKU017,2,2,1.000000,0.127907,0.639535,0.377076
29,SKU002,SKU004,2,2,1.000000,0.081395,0.406977,0.343854
625,SKU024,SKU005,3,2,0.666667,0.127907,0.639535,0.329942
253,SKU010,SKU012,3,2,0.666667,0.104651,0.523256,0.315407
532,SKU020,SKU021,3,2,0.666667,0.104651,0.523256,0.315407
2,SKU001,SKU004,3,2,0.666667,0.081395,0.406977,0.300872
81,SKU004,SKU001,3,2,0.666667,0.081395,0.406977,0.300872
334,SKU013,SKU011,4,2,0.500000,0.127907,0.639535,0.293282
82,SKU004,SKU002,3,2,0.666667,0.058140,0.290698,0.286337
306,SKU012,SKU010,4,2,0.500000,0.081395,0.406977,0.267442


In [43]:
# Validación de las probabilidades posteriores

assert len(asociaciones_df) == 756

assert asociaciones_df["media_posterior"].between(0, 1).all()

assert asociaciones_df["media_posterior"].notna().all()

assert (
    asociaciones_df["sku_origen"]
    != asociaciones_df["sku_recomendado"]
).all()

print("Validación completada correctamente")
print("Relaciones:", len(asociaciones_df))
print(
    "Probabilidad mínima:",
    round(asociaciones_df["media_posterior"].min(), 4)
)
print(
    "Probabilidad máxima:",
    round(asociaciones_df["media_posterior"].max(), 4)
)

Validación completada correctamente
Relaciones: 756
Probabilidad mínima: 0.0058
Probabilidad máxima: 0.3771


# Validación de consistencia del modelo bayesiano

Se calcularon las medias posteriores para las 756 relaciones dirigidas posibles entre los 28 productos.

Se realizaron las siguientes comprobaciones:

- El número de relaciones es igual a 756.
- Todas las medias posteriores pertenecen al intervalo \([0,1]\).
- No existen valores faltantes en las medias posteriores.
- Ningún producto se relaciona consigo mismo.

Los valores obtenidos fueron:

- Media posterior mínima: 0.0058.
- Media posterior máxima: 0.3771.

Todas las comprobaciones se completaron satisfactoriamente.

Estas pruebas verifican la consistencia computacional de las estimaciones, pero no constituyen una evaluación de su capacidad predictiva ni demuestran que las recomendaciones aumenten las ventas.

El modelo bayesiano representa uno de los componentes estadísticos del sistema. La selección final de recomendaciones también deberá considerar la pertinencia de los productos, la disponibilidad del inventario y las restricciones comerciales.

# 5 Guardar las estimaciones del modelo

In [45]:
# 4.3 Exportación de las estimaciones del modelo

from pathlib import Path

ruta_salida = Path(
    r"C:\Users\Lucinda\Downloads\Prueba tecnica\Data\asociaciones_modelo.csv"
)

asociaciones_df.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig"
)

print("Archivo guardado en:", ruta_salida)
print("Relaciones exportadas:", len(asociaciones_df))
print("¿Existe el archivo?:", ruta_salida.exists())

Archivo guardado en: C:\Users\Lucinda\Downloads\Prueba tecnica\Data\asociaciones_modelo.csv
Relaciones exportadas: 756
¿Existe el archivo?: True


In [46]:
from pathlib import Path

proyecto = Path(r"C:\Users\Lucinda\Downloads\Prueba tecnica")

archivos = {
    "Catálogo de productos": proyecto / "Data" / "products.csv",
    "Historial de ventas": proyecto / "Data" / "sales.csv",
    "Modelo bayesiano": proyecto / "Data" / "asociaciones_modelo.csv",
    "API FastAPI": proyecto / "backend" / "main.py",
}

for nombre, ruta in archivos.items():
    print(f"{nombre}: {'OK' if ruta.is_file() else 'NO ENCONTRADO'}")

print(
    "Carpeta frontend:",
    "OK" if (proyecto / "frontend").is_dir() else "NO ENCONTRADA"
)

Catálogo de productos: OK
Historial de ventas: OK
Modelo bayesiano: OK
API FastAPI: OK
Carpeta frontend: OK


# 6 Implementación del sistema de recomendaciones e inventario

## 1 Arquitectura general del sistema

En las secciones anteriores se desarrolló un modelo estadístico para estimar las relaciones entre los productos del catálogo, utilizando el historial de ventas y un procedimiento de suavizado bayesiano.

Las 756 relaciones dirigidas calculadas se exportaron al archivo `asociaciones_modelo.csv`, con el propósito de utilizar sus estimaciones en una aplicación funcional.

Sin embargo, el modelo estadístico representa solamente una parte de la solución. Para atender los requerimientos del negocio, es necesario integrar los siguientes componentes:

1. **Base de datos (SQLite):** almacenará el catálogo de productos y las existencias disponibles. El inventario será compartido entre las cinco sucursales: Cancún, Chihuahua, Ciudad de México, Monterrey y Mérida.

2. **Backend (Python y FastAPI):** proporcionará los servicios necesarios para consultar, crear, actualizar y eliminar productos, registrar compras y generar recomendaciones.

3. **Motor de recomendaciones:** utilizará las relaciones estimadas, junto con criterios de pertinencia y disponibilidad, para seleccionar productos complementarios a los artículos del carrito.

4. **Frontend (React):** permitirá interactuar con el catálogo, seleccionar una sucursal, administrar el carrito y consultar recomendaciones antes de confirmar una compra.

###  Integración del modelo matemático con la aplicación

El modelo bayesiano proporciona estimaciones de asociación entre pares de productos, pero no determina por sí solo qué artículos deben recomendarse.

La aplicación deberá incorporar restricciones adicionales, particularmente:

- Excluir productos que ya se encuentran en el carrito.
- Recomendar únicamente artículos con inventario disponible.
- Permitir la administración de relaciones comerciales entre productos.
- Limitar la cantidad de recomendaciones presentadas.

###  Control del inventario compartido

Aunque cada compra se asociará con una sucursal específica, las existencias se administrarán de manera centralizada.

Por lo tanto, una venta realizada en cualquier sucursal deberá disminuir el inventario general. El sistema también deberá impedir que se registren compras cuya cantidad solicitada supere las existencias disponibles.

Para implementar estas operaciones se utilizará SQLite, mientras que FastAPI proporcionará los servicios de acceso y actualización de la información.

###  Estrategia de implementación

El desarrollo se realizará de forma incremental, comenzando por la estructura de datos y continuando con los servicios del backend, el motor de recomendaciones y la interfaz de usuario.

Cada componente se verificará mediante pruebas de funcionamiento antes de integrarlo con los demás.

#  Diseño de la base de datos

Antes de construir la base de datos SQLite, se revisará la estructura del catálogo original para identificar los campos disponibles y sus tipos de datos.

Esta revisión permitirá definir correctamente la tabla de productos, conservar la información proporcionada y establecer las restricciones necesarias para administrar el inventario compartido.

En particular, se verificará que los precios y las existencias tengan tipos numéricos adecuados para las operaciones de compra.

In [47]:
#  Exploración de la estructura del catálogo

print("Columnas del catálogo:")
print(products.columns.tolist())

print("\nTipos de datos:")
print(products.dtypes)

print("\nPrimeros tres productos:")
display(products.head(3))

Columnas del catálogo:
['sku', 'nombre', 'descripcion', 'categoria', 'material', 'uso_recomendado', 'precio', 'stock', 'texto']

Tipos de datos:
sku                object
nombre             object
descripcion        object
categoria          object
material           object
uso_recomendado    object
precio              int64
stock               int64
texto              object
dtype: object

Primeros tres productos:


,sku,nombre,descripcion,categoria,material,uso_recomendado,precio,stock,texto
0,SKU001,Soplete de gas MAPP,Soplete portátil para soldadura y corte menor ...,herramienta,acero y latón,soldadura y corte menor,450,15,Soplete de gas MAPP Soplete portátil para sold...
1,SKU002,Varilla de soldar estaño 60/40,Varilla de aporte para soldadura de baja tempe...,consumible,estaño-plomo,soldadura eléctrica y electrónica,80,40,Varilla de soldar estaño 60/40 Varilla de apor...
2,SKU003,Varilla de soldar plata,Varilla de aporte para uniones de alta resiste...,consumible,aleación de plata,soldadura de alta resistencia,220,10,Varilla de soldar plata Varilla de aporte para...


# Diseño de la tabla de productos

A partir de la exploración del catálogo, se identificaron ocho campos originales que deberán almacenarse en la base de datos:

| Campo | Tipo SQLite | Descripción |
|---|---|---|
| sku | TEXT | Identificador único del producto |
| nombre | TEXT | Nombre comercial |
| descripcion | TEXT | Descripción del producto |
| categoria | TEXT | Categoría comercial |
| material | TEXT | Material principal |
| uso_recomendado | TEXT | Aplicación o uso sugerido |
| precio | INTEGER | Precio unitario |
| stock | INTEGER | Existencias disponibles |

La columna `texto`, utilizada previamente para calcular similitudes mediante TF-IDF, es una variable derivada del análisis y no se incorporará a la tabla de inventario.

**Restricciones de integridad:**

- `sku` será la clave primaria y no podrá repetirse.
- `precio` deberá ser mayor o igual que cero.
- `stock` deberá ser mayor o igual que cero.

**Inventario centralizado:**

La tabla almacenará una única cantidad disponible por producto, independientemente de la sucursal que realice una venta. Esto permitirá administrar las existencias como un recurso compartido entre las cinco tiendas.

Las restricciones de la base de datos protegerán la integridad de los valores almacenados. Adicionalmente, las operaciones de compra deberán ejecutarse mediante transacciones que comprueben la disponibilidad antes de descontar las existencias, evitando ventas superiores al inventario disponible.

In [48]:
#  Creación de la tabla de productos en SQLite

from pathlib import Path

proyecto = Path(r"C:\Users\Lucinda\Downloads\Prueba tecnica")
archivo_database = proyecto / "backend" / "database.py"

codigo_database = '''
import sqlite3
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent
DB_PATH = BASE_DIR / "Data" / "ferreteria.db"


def conectar():
    conexion = sqlite3.connect(DB_PATH, timeout=30)
    conexion.row_factory = sqlite3.Row
    return conexion


def crear_tablas():
    with conectar() as conexion:
        conexion.execute("""
            CREATE TABLE IF NOT EXISTS productos (
                sku TEXT PRIMARY KEY,
                nombre TEXT NOT NULL,
                descripcion TEXT,
                categoria TEXT,
                material TEXT,
                uso_recomendado TEXT,
                precio INTEGER NOT NULL CHECK (precio >= 0),
                stock INTEGER NOT NULL CHECK (stock >= 0)
            )
        """)
        conexion.commit()
'''

archivo_database.write_text(
    codigo_database,
    encoding="utf-8"
)

print("Archivo creado:", archivo_database)
print("¿Existe?:", archivo_database.exists())

Archivo creado: C:\Users\Lucinda\Downloads\Prueba tecnica\backend\database.py
¿Existe?: True


In [49]:
#  Creación y verificación de la base de datos SQLite

import sys
import sqlite3
from pathlib import Path

proyecto = Path(r"C:\Users\Lucinda\Downloads\Prueba tecnica")
carpeta_backend = proyecto / "backend"

if str(carpeta_backend) not in sys.path:
    sys.path.insert(0, str(carpeta_backend))

from database import crear_tablas, DB_PATH

# Crear la tabla de productos
crear_tablas()

# Verificar su estructura
with sqlite3.connect(DB_PATH) as conexion:
    estructura = conexion.execute(
        "PRAGMA table_info(productos)"
    ).fetchall()

print("Base de datos:", DB_PATH)
print("¿Existe?:", DB_PATH.exists())

print("\nColumnas de la tabla productos:")
for columna in estructura:
    print(columna[1], "-", columna[2])

Base de datos: C:\Users\Lucinda\Downloads\Prueba tecnica\Data\ferreteria.db
¿Existe?: True

Columnas de la tabla productos:
sku - TEXT
nombre - TEXT
descripcion - TEXT
categoria - TEXT
material - TEXT
uso_recomendado - TEXT
precio - INTEGER
stock - INTEGER


#  Carga inicial del inventario

Una vez creada la estructura de la base de datos SQLite, se procederá a incorporar los productos del catálogo original `products.csv`.

El objetivo es establecer el inventario inicial del sistema, conservando la información proporcionada para cada producto, incluyendo su identificador, descripción, categoría, precio y existencias.

Para esta operación se utilizarán los siguientes criterios:

1. Cada producto se identificará mediante su SKU, que constituye la clave primaria de la tabla.
2. Se conservarán los valores originales de precio y stock.
3. Se verificará que los 28 productos se encuentren registrados correctamente.
4. La carga inicial no deberá duplicar productos ni restablecer existencias cuando se ejecute nuevamente.

**Consideración sobre el inventario compartido**

Las existencias iniciales representan el inventario centralizado disponible para las cinco sucursales. Las compras posteriores deberán descontarse de esta misma tabla.

El historial `sales.csv` se utilizará para el análisis de recomendaciones, pero no se descontará nuevamente del inventario inicial, ya que no se ha establecido que las existencias del catálogo correspondan a un momento anterior a esas ventas.

Esta separación evita introducir descuentos de inventario que no estén justificados por los datos proporcionados.

In [50]:
#  Carga inicial del catálogo en SQLite

import sqlite3
import pandas as pd

# Seleccionamos únicamente las columnas originales
columnas = [
    "sku",
    "nombre",
    "descripcion",
    "categoria",
    "material",
    "uso_recomendado",
    "precio",
    "stock"
]

catalogo = products[columnas].copy()

# Convertimos los registros al formato de SQLite
registros = [
    (
        str(fila.sku),
        str(fila.nombre),
        str(fila.descripcion),
        str(fila.categoria),
        str(fila.material),
        str(fila.uso_recomendado),
        int(fila.precio),
        int(fila.stock)
    )
    for fila in catalogo.itertuples(index=False)
]

# Insertamos los productos sin sobrescribir registros existentes
with sqlite3.connect(DB_PATH) as conexion:

    conexion.executemany(
        """
        INSERT OR IGNORE INTO productos (
            sku,
            nombre,
            descripcion,
            categoria,
            material,
            uso_recomendado,
            precio,
            stock
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        registros
    )

    conexion.commit()

    total_productos = conexion.execute(
        "SELECT COUNT(*) FROM productos"
    ).fetchone()[0]

print("Productos en el catálogo original:", len(catalogo))
print("Productos registrados en SQLite:", total_productos)

Productos en el catálogo original: 28
Productos registrados en SQLite: 28


In [51]:
# Validación del inventario inicial

import sqlite3
import pandas as pd

# Consultamos los productos almacenados en SQLite
with sqlite3.connect(DB_PATH) as conexion:
    inventario_db = pd.read_sql_query(
        "SELECT sku, precio, stock FROM productos",
        conexion
    )

# Comparamos con el catálogo original
comparacion = products[
    ["sku", "precio", "stock"]
].merge(
    inventario_db,
    on="sku",
    how="outer",
    suffixes=("_original", "_sqlite"),
    indicator=True
)

# Verificamos que todos los productos y valores coincidan
productos_coinciden = (comparacion["_merge"] == "both").all()

precios_coinciden = (
    comparacion["precio_original"]
    == comparacion["precio_sqlite"]
).all()

stocks_coinciden = (
    comparacion["stock_original"]
    == comparacion["stock_sqlite"]
).all()

print("Productos encontrados en ambas fuentes:", productos_coinciden)
print("Precios coinciden:", precios_coinciden)
print("Existencias coinciden:", stocks_coinciden)
print("Productos comparados:", len(comparacion))

Productos encontrados en ambas fuentes: True
Precios coinciden: True
Existencias coinciden: True
Productos comparados: 28


#  Registro de compras y control del inventario compartido

Una vez validada la carga inicial de los 28 productos, se implementará el registro de compras.

El sistema deberá permitir que un cliente seleccione una sucursal y adquiera uno o varios productos del catálogo. Todas las compras utilizarán el mismo inventario centralizado.

###  Reglas de negocio

Para registrar una compra se deberán cumplir las siguientes condiciones:

1. La sucursal seleccionada deberá pertenecer al conjunto de cinco sucursales autorizadas: Cancún, Chihuahua, Ciudad de México, Monterrey y Mérida.
2. Todos los productos solicitados deberán existir en el catálogo.
3. Las cantidades compradas deberán ser números enteros positivos.
4. La cantidad solicitada de cada producto no podrá superar sus existencias disponibles.
5. Una compra deberá descontar las existencias del inventario centralizado.
6. Si alguno de los productos no puede adquirirse, la compra completa deberá cancelarse sin modificar el inventario.

###  Integridad de las transacciones

Para evitar inconsistencias, las compras se procesarán mediante transacciones de SQLite.

Una transacción garantiza que un conjunto de operaciones se confirme completamente o se revierta en caso de error.

Se utilizará `BEGIN IMMEDIATE` para obtener el bloqueo de escritura antes de procesar la compra. Cada descuento se realizará mediante una actualización condicionada a la disponibilidad del producto.

La operación solamente se confirmará cuando todos los productos solicitados hayan sido descontados correctamente.

Este procedimiento permitirá evitar que dos compras simultáneas descuenten las mismas existencias y que una compra parcialmente procesada deje el inventario inconsistente.

###  Trazabilidad de las compras

Además de actualizar las existencias, se almacenará información de cada compra:

- Identificador único de la compra.
- Sucursal donde se realizó.
- Fecha y hora.
- Productos adquiridos.
- Cantidades y precios unitarios al momento de la compra.

Esta información permitirá consultar las operaciones realizadas y mantener un historial independiente de las ventas históricas proporcionadas originalmente.

In [52]:
# Creación de las tablas de compras

import sqlite3

with sqlite3.connect(DB_PATH) as conexion:

    # Tabla principal: una fila por compra
    conexion.execute("""
        CREATE TABLE IF NOT EXISTS compras (
            id_compra INTEGER PRIMARY KEY AUTOINCREMENT,
            sucursal TEXT NOT NULL CHECK (
                sucursal IN (
                    'Cancún',
                    'Chihuahua',
                    'Ciudad de México',
                    'Monterrey',
                    'Mérida'
                )
            ),
            fecha TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
        )
    """)

    # Detalle: productos incluidos en cada compra
    conexion.execute("""
        CREATE TABLE IF NOT EXISTS detalle_compras (
            id_compra INTEGER NOT NULL,
            sku TEXT NOT NULL,
            cantidad INTEGER NOT NULL CHECK (cantidad > 0),
            precio_unitario INTEGER NOT NULL CHECK (precio_unitario >= 0),

            PRIMARY KEY (id_compra, sku),

            FOREIGN KEY (id_compra)
                REFERENCES compras(id_compra),

            FOREIGN KEY (sku)
                REFERENCES productos(sku)
        )
    """)

    conexion.commit()

    # Verificamos que las tablas existan
    tablas = conexion.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name IN ('productos', 'compras', 'detalle_compras')
        ORDER BY name
    """).fetchall()

print("Tablas creadas:")
for tabla in tablas:
    print("-", tabla[0])

Tablas creadas:
- compras
- detalle_compras
- productos


In [53]:
#  Actualización de la inicialización de SQLite

from pathlib import Path

archivo_database = Path(
    r"C:\Users\Lucinda\Downloads\Prueba tecnica\backend\database.py"
)

codigo_database = '''
import sqlite3
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent
DB_PATH = BASE_DIR / "Data" / "ferreteria.db"


def conectar():
    conexion = sqlite3.connect(DB_PATH, timeout=30)
    conexion.row_factory = sqlite3.Row
    conexion.execute("PRAGMA foreign_keys = ON")
    return conexion


def crear_tablas():
    with conectar() as conexion:

        conexion.execute("""
            CREATE TABLE IF NOT EXISTS productos (
                sku TEXT PRIMARY KEY,
                nombre TEXT NOT NULL,
                descripcion TEXT,
                categoria TEXT,
                material TEXT,
                uso_recomendado TEXT,
                precio INTEGER NOT NULL CHECK (precio >= 0),
                stock INTEGER NOT NULL CHECK (stock >= 0)
            )
        """)

        conexion.execute("""
            CREATE TABLE IF NOT EXISTS compras (
                id_compra INTEGER PRIMARY KEY AUTOINCREMENT,
                sucursal TEXT NOT NULL CHECK (
                    sucursal IN (
                        'Cancún',
                        'Chihuahua',
                        'Ciudad de México',
                        'Monterrey',
                        'Mérida'
                    )
                ),
                fecha TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
            )
        """)

        conexion.execute("""
            CREATE TABLE IF NOT EXISTS detalle_compras (
                id_compra INTEGER NOT NULL,
                sku TEXT NOT NULL,
                cantidad INTEGER NOT NULL CHECK (cantidad > 0),
                precio_unitario INTEGER NOT NULL CHECK (precio_unitario >= 0),

                PRIMARY KEY (id_compra, sku),

                FOREIGN KEY (id_compra)
                    REFERENCES compras(id_compra),

                FOREIGN KEY (sku)
                    REFERENCES productos(sku)
            )
        """)

        conexion.commit()
'''

archivo_database.write_text(
    codigo_database,
    encoding="utf-8"
)

print("Archivo actualizado:", archivo_database)
print("¿Existe?:", archivo_database.exists())

Archivo actualizado: C:\Users\Lucinda\Downloads\Prueba tecnica\backend\database.py
¿Existe?: True


In [54]:
# Verificar que conservamos el inventario
import sqlite3

with sqlite3.connect(DB_PATH) as conexion:

    total_productos = conexion.execute(
        "SELECT COUNT(*) FROM productos"
    ).fetchone()[0]

    total_stock = conexion.execute(
        "SELECT SUM(stock) FROM productos"
    ).fetchone()[0]

    tablas = conexion.execute("""
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name IN ('productos', 'compras', 'detalle_compras')
        ORDER BY name
    """).fetchall()

print("Productos conservados:", total_productos)
print("Existencias totales:", total_stock)
print("Tablas existentes:", [tabla[0] for tabla in tablas])

Productos conservados: 28
Existencias totales: 1958
Tablas existentes: ['compras', 'detalle_compras', 'productos']


# Preparar una copia de seguridad
Antes de modificar nuevamente backend/database.py, vamos a guardar una copia del código actual. Así evitaremos el problema que tuvimos cuando se sobrescribió el archivo.

In [55]:
# 5.3.6 Copia de seguridad del módulo de base de datos

from pathlib import Path
import shutil

proyecto = Path(r"C:\Users\Lucinda\Downloads\Prueba tecnica")

archivo_original = proyecto / "backend" / "database.py"
archivo_respaldo = proyecto / "backend" / "database_backup.py"

shutil.copy2(archivo_original, archivo_respaldo)

print("Copia de seguridad creada:", archivo_respaldo)
print("¿Existe?:", archivo_respaldo.exists())

Copia de seguridad creada: C:\Users\Lucinda\Downloads\Prueba tecnica\backend\database_backup.py
¿Existe?: True


La siguiente función deberá realizar tres operaciones dentro de una sola transacción: registrar la compra, guardar sus productos y descontar las existencias.

Si algún producto no tiene suficiente inventario, se cancelará toda la operación.

In [56]:
# 5.3.7 Implementación del registro de compras

from pathlib import Path

archivo_database = Path(
    r"C:\Users\Lucinda\Downloads\Prueba tecnica\backend\database.py"
)

codigo_compras = '''

SUCURSALES = (
    "Cancún",
    "Chihuahua",
    "Ciudad de México",
    "Monterrey",
    "Mérida"
)


def registrar_compra(sucursal, articulos):
    """
    Registra una compra y descuenta el inventario compartido.

    articulos: lista de diccionarios:
        [{"sku": "SKU001", "cantidad": 2}]
    """

    if sucursal not in SUCURSALES:
        raise ValueError("Sucursal no válida")

    if not articulos:
        raise ValueError("La compra debe contener productos")

    # Agrupamos SKU repetidos antes de descontar existencias
    cantidades = {}

    for articulo in articulos:
        sku = articulo["sku"]
        cantidad = articulo["cantidad"]

        if not isinstance(cantidad, int) or isinstance(cantidad, bool):
            raise ValueError("La cantidad debe ser un entero")

        if cantidad <= 0:
            raise ValueError("La cantidad debe ser positiva")

        cantidades[sku] = cantidades.get(sku, 0) + cantidad

    conexion = conectar()

    try:
        # Reservamos el acceso de escritura
        conexion.execute("BEGIN IMMEDIATE")

        cursor = conexion.execute(
            "INSERT INTO compras (sucursal) VALUES (?)",
            (sucursal,)
        )

        id_compra = cursor.lastrowid
        total = 0

        for sku, cantidad in cantidades.items():

            producto = conexion.execute(
                "SELECT precio FROM productos WHERE sku = ?",
                (sku,)
            ).fetchone()

            if producto is None:
                raise ValueError(f"Producto inexistente: {sku}")

            precio = producto["precio"]

            # Solo descuenta si existe inventario suficiente
            resultado = conexion.execute(
                """
                UPDATE productos
                SET stock = stock - ?
                WHERE sku = ? AND stock >= ?
                """,
                (cantidad, sku, cantidad)
            )

            if resultado.rowcount != 1:
                raise ValueError(
                    f"Inventario insuficiente para {sku}"
                )

            conexion.execute(
                """
                INSERT INTO detalle_compras
                    (id_compra, sku, cantidad, precio_unitario)
                VALUES (?, ?, ?, ?)
                """,
                (id_compra, sku, cantidad, precio)
            )

            total += precio * cantidad

        conexion.commit()

        return {
            "id_compra": id_compra,
            "sucursal": sucursal,
            "total": total,
            "articulos": cantidades
        }

    except Exception:
        conexion.rollback()
        raise

    finally:
        conexion.close()
'''

contenido_actual = archivo_database.read_text(encoding="utf-8")

if "def registrar_compra(" not in contenido_actual:
    with archivo_database.open("a", encoding="utf-8") as archivo:
        archivo.write(codigo_compras)
    print("Función registrar_compra agregada correctamente")
else:
    print("La función ya existe. No se realizaron cambios")

print("Archivo:", archivo_database)

Función registrar_compra agregada correctamente
Archivo: C:\Users\Lucinda\Downloads\Prueba tecnica\backend\database.py


# Probar una compra imposible

In [57]:
#Prueba de protección contra sobreventas

import importlib
import database

# Recargamos el módulo porque acabamos de modificar database.py
importlib.reload(database)

# Consultamos el inventario antes de la prueba
with database.conectar() as conexion:
    stock_antes = conexion.execute(
        "SELECT stock FROM productos WHERE sku = ?",
        ("SKU001",)
    ).fetchone()["stock"]

print("Stock antes de la prueba:", stock_antes)

# Intentamos comprar una cantidad superior a la disponible
try:
    database.registrar_compra(
        sucursal="Cancún",
        articulos=[
            {"sku": "SKU001", "cantidad": stock_antes + 1}
        ]
    )
    print("ERROR: La compra fue aceptada indebidamente")

except ValueError as error:
    print("Compra rechazada correctamente:", error)

# Comprobamos que el inventario no cambió
with database.conectar() as conexion:
    stock_despues = conexion.execute(
        "SELECT stock FROM productos WHERE sku = ?",
        ("SKU001",)
    ).fetchone()["stock"]

print("Stock después de la prueba:", stock_despues)
print("¿Inventario intacto?:", stock_antes == stock_despues)

Stock antes de la prueba: 15
Compra rechazada correctamente: Inventario insuficiente para SKU001
Stock después de la prueba: 15
¿Inventario intacto?: True


# Probar una compra válida con dos productos

In [59]:
# 5.3.9 Prueba de una compra válida con varios productos

skus_prueba = ["SKU001", "SKU002"]

# Consultamos las existencias antes de comprar
with database.conectar() as conexion:
    stock_antes = {
        fila["sku"]: fila["stock"]
        for fila in conexion.execute(
            "SELECT sku, stock FROM productos WHERE sku IN (?, ?)",
            skus_prueba
        ).fetchall()
    }

print("Inventario antes:", stock_antes)

# Registramos la compra
compra = database.registrar_compra(
    sucursal="Monterrey",
    articulos=[
        {"sku": "SKU001", "cantidad": 1},
        {"sku": "SKU002", "cantidad": 2}
    ]
)

print("\nCompra registrada:", compra)

# Consultamos las existencias después
with database.conectar() as conexion:
    stock_despues = {
        fila["sku"]: fila["stock"]
        for fila in conexion.execute(
            "SELECT sku, stock FROM productos WHERE sku IN (?, ?)",
            skus_prueba
        ).fetchall()
    }

print("\nInventario después:", stock_despues)

# Verificamos los descuentos
assert stock_despues["SKU001"] == stock_antes["SKU001"] - 1
assert stock_despues["SKU002"] == stock_antes["SKU002"] - 2

print("\nValidación: ambos descuentos son correctos")

Inventario antes: {'SKU001': 15, 'SKU002': 40}

Compra registrada: {'id_compra': 1, 'sucursal': 'Monterrey', 'total': 610, 'articulos': {'SKU001': 1, 'SKU002': 2}}

Inventario después: {'SKU001': 14, 'SKU002': 38}

Validación: ambos descuentos son correctos


# Pruebas de funcionamiento del registro de compras

Para verificar el funcionamiento del módulo de compras, se realizaron dos pruebas sobre la base de datos SQLite.

**Prueba 1. Rechazo de una compra con inventario insuficiente**

Se intentó comprar una cantidad del producto SKU001 superior a sus existencias disponibles.

Resultados:

- Existencias iniciales: 15 unidades.
- La operación fue rechazada con el mensaje `Inventario insuficiente para SKU001`.
- Existencias posteriores: 15 unidades.
- No se produjo ningún descuento de inventario.

Esta prueba confirma que el sistema rechaza una compra individual cuando la cantidad solicitada supera las existencias disponibles.

**Prueba 2. Registro de una compra válida**

Se registró una compra en la sucursal Monterrey con los siguientes artículos:

| Producto | Cantidad | Precio unitario | Importe |
|---|---:|---:|---:|
| SKU001 | 1 | $450 | $450 |
| SKU002 | 2 | $80 | $160 |
| **Total** | **3** | | **$610** |

La operación recibió el identificador `id_compra = 1`.

Los cambios observados en el inventario fueron:

| Producto | Stock anterior | Stock posterior |
|---|---:|---:|
| SKU001 | 15 | 14 |
| SKU002 | 40 | 38 |

Los descuentos coincidieron exactamente con las cantidades compradas.



Estas pruebas todavía no verifican todos los escenarios posibles, particularmente la reversión de una compra con varios productos cuando falla uno de ellos, ni el comportamiento ante solicitudes concurrentes.

La implementación utiliza transacciones de SQLite y actualizaciones condicionadas para controlar estos casos, pero será necesario realizar pruebas adicionales.



# 7 Desarrollo de la API REST con FastAPI

##  Objetivo y arquitectura

La API REST permitirá que la interfaz web se comunique con la base de datos SQLite y con el motor de recomendaciones.

Se utilizará FastAPI, un framework de Python que facilita la construcción de servicios HTTP y proporciona documentación interactiva de los endpoints.

La arquitectura separará tres responsabilidades:

- **Persistencia:** `database.py` administrará las conexiones, las tablas y las operaciones sobre SQLite.
- **Servicios HTTP:** `main.py` recibirá las solicitudes, validará los datos y devolverá las respuestas.
- **Recomendaciones:** un módulo independiente consultará las asociaciones estimadas y aplicará los criterios de selección.

##  Endpoints previstos

| Método | Ruta | Función |
|---|---|---|
| GET | `/productos` | Consultar el catálogo y el inventario |
| GET | `/productos/{sku}` | Consultar un producto |
| POST | `/productos` | Crear un producto |
| PUT | `/productos/{sku}` | Actualizar un producto |
| DELETE | `/productos/{sku}` | Eliminar un producto |
| POST | `/compras` | Registrar una compra y descontar existencias |
| GET | `/recomendaciones/{sku}` | Obtener recomendaciones para un producto |
| GET | `/relaciones` | Consultar las relaciones entre productos |
| PUT | `/relaciones` | Ajustar relaciones comerciales |

La implementación se realizará progresivamente y cada endpoint se probará antes de integrarlo con la interfaz.

##  Criterios de funcionamiento

La API deberá consultar siempre el inventario actualizado en SQLite, no el archivo CSV original.

Las operaciones de compra utilizarán la función transaccional desarrollada anteriormente, con el objetivo de impedir ventas superiores a las existencias disponibles.

Las recomendaciones se presentarán antes de confirmar una compra, de modo que el cliente pueda agregar artículos complementarios al carrito.

La sucursal seleccionada se utilizará para registrar la compra, mientras que las existencias y las asociaciones iniciales se administrarán de manera centralizada.

In [61]:
#  Revisión del archivo principal de FastAPI

from pathlib import Path

archivo_main = Path(
    r"C:\Users\Lucinda\Downloads\Prueba tecnica\backend\main.py"
)

print("¿Existe el archivo?:", archivo_main.exists())

if archivo_main.exists():
    print("\nContenido actual de main.py:\n")
    print(archivo_main.read_text(encoding="utf-8"))

¿Existe el archivo?: True

Contenido actual de main.py:


from fastapi import FastAPI

app = FastAPI(
    title="API Ferretería",
    description="Sistema de inventario y recomendaciones",
    version="1.0.0"
)

@app.get("/")
def inicio():
    return {
        "mensaje": "Bienvenido a la API de Ferretería",
        "estado": "funcionando"
    }



# 8 Desarrollo de la API REST con FastAPI

###  Objetivo

En esta etapa se desarrolla una API REST utilizando **FastAPI**, un framework de Python que permite comunicar la lógica del sistema con una aplicación web.

La API funcionará como intermediaria entre la interfaz del usuario, la base de datos SQLite y el sistema de recomendaciones.

###  Funcionalidades previstas

La API permitirá:

1. Consultar los productos disponibles, sus precios y existencias.
2. Crear, actualizar y eliminar productos del catálogo.
3. Registrar compras realizadas desde cualquiera de las cinco sucursales.
4. Actualizar el inventario compartido después de cada compra.
5. Evitar compras que excedan las existencias disponibles.
6. Generar recomendaciones de productos relacionados a partir del modelo desarrollado.
7. Permitir la consulta y administración de las relaciones entre productos.

###  Arquitectura del sistema

El sistema se organizará en tres componentes principales:

- **Frontend (React):** interfaz desde la cual el usuario seleccionará una sucursal, consultará productos y realizará compras.
- **Backend (FastAPI):** recibirá las solicitudes, aplicará las reglas de negocio y devolverá las respuestas.
- **Base de datos (SQLite):** almacenará el catálogo, el inventario compartido y las compras registradas.

El modelo de recomendaciones se integrará al backend para sugerir productos complementarios antes de finalizar una compra.

###  Consideraciones de implementación

El inventario será único para las cinco sucursales. Por lo tanto, una compra realizada en cualquier sucursal reducirá las existencias disponibles para todas.

Las operaciones de compra deberán ser transaccionales: si alguno de los productos solicitados no tiene existencias suficientes, la compra completa se rechazará sin modificar el inventario.

Se realizarán pruebas de los endpoints para comprobar su funcionamiento y validar las reglas de negocio.

In [65]:
#Antes de continuar vamos a poner este script en la misma carpeta donde estan nuestros datos


In [64]:
import os

os.chdir(r"C:\Users\Lucinda\Downloads\Prueba tecnica")
print(os.getcwd())

C:\Users\Lucinda\Downloads\Prueba tecnica


In [66]:
from pathlib import Path

ruta = Path("backend/main.py")
print(ruta.read_text(encoding="utf-8"))


from fastapi import FastAPI

app = FastAPI(
    title="API Ferretería",
    description="Sistema de inventario y recomendaciones",
    version="1.0.0"
)

@app.get("/")
def inicio():
    return {
        "mensaje": "Bienvenido a la API de Ferretería",
        "estado": "funcionando"
    }



Crear el endpoint GET /productos
Este endpoint permitirá que la aplicación web solicite el catálogo completo de la ferretería, incluyendo precios y existencias.


In [67]:
from pathlib import Path

ruta = Path("backend/main.py")

codigo = ruta.read_text(encoding="utf-8")

nuevo_codigo = """

from database import conectar

@app.get("/productos")
def obtener_productos():
    with conectar() as conexion:
        productos = conexion.execute(
            "SELECT * FROM productos ORDER BY sku"
        ).fetchall()

    return [dict(producto) for producto in productos]
"""

if '@app.get("/productos")' not in codigo:
    ruta.write_text(codigo + nuevo_codigo, encoding="utf-8")
    print("Endpoint GET /productos agregado correctamente.")
else:
    print("El endpoint ya existe. No se modificó el archivo.")

Endpoint GET /productos agregado correctamente.


# Pruebas iniciales de la API REST

Se inició el servidor utilizando Uvicorn desde Anaconda PowerShell y se realizaron pruebas mediante la documentación interactiva de FastAPI (`/docs`).

**Prueba 1. Endpoint GET /**

- Código de respuesta: HTTP 200.
- Resultado: mensaje de bienvenida y estado del servidor.
- Conclusión: el servidor responde correctamente.

**Prueba 2. Endpoint GET /productos**

- Código de respuesta: HTTP 200.
- Resultado: catálogo de productos en formato JSON.
- Se verificaron los campos SKU, nombre, descripción, categoría, material, uso recomendado, precio y stock.
- Las existencias reflejan las compras de prueba realizadas previamente.

 Se verificó la conexión entre FastAPI y SQLite para consultar el inventario compartido.


In [69]:
#Antes de continuar vamos a guardar ete notebook en Git Hub
from pathlib import Path
import shutil

origen = Path(r"C:\Users\Lucinda\Documents\01_exploracion_datos.ipynb")
destino = Path(r"C:\Users\Lucinda\Downloads\Prueba tecnica\notebooks\01_exploracion_datos.ipynb")

if not origen.exists():
    print("No se encontró el notebook original.")
elif destino.exists():
    print("Ya existe una copia en el proyecto. No se sobrescribió.")
else:
    destino.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(origen, destino)
    print("Notebook copiado correctamente al proyecto.")

Notebook copiado correctamente al proyecto.


#  Registro de compras mediante POST /compras

El endpoint `POST /compras` permitirá registrar una compra indicando la sucursal y los productos solicitados.

**Datos de entrada:**
- Sucursal donde se realiza la compra.
- Lista de productos identificados por SKU.
- Cantidad solicitada de cada producto.

**Reglas de negocio:**
1. La sucursal debe pertenecer a las cinco sucursales autorizadas.
2. Las cantidades deben ser números enteros positivos.
3. Todos los productos deben existir en el catálogo.
4. El inventario es compartido entre las sucursales.
5. Si algún producto no tiene existencias suficientes, se rechaza toda la compra.
6. Si la compra es válida, se descuentan las existencias y se registra la operación en SQLite.

La implementación utilizará la función `registrar_compra()` previamente desarrollada en `database.py`, que emplea una transacción para evitar modificaciones parciales del inventario.

**Pruebas previstas:** compra válida, stock insuficiente, SKU inexistente y sucursal no válida.

In [70]:
from pathlib import Path

ruta = Path(r"C:\Users\Lucinda\Downloads\Prueba tecnica\backend\main.py")

codigo = ruta.read_text(encoding="utf-8")

nuevo_codigo = """

from pydantic import BaseModel, Field
from fastapi import HTTPException
from database import registrar_compra, SUCURSALES

class ArticuloCompra(BaseModel):
    sku: str
    cantidad: int = Field(gt=0, strict=True)

class SolicitudCompra(BaseModel):
    sucursal: str
    articulos: list[ArticuloCompra]

@app.post("/compras", status_code=201)
def crear_compra(solicitud: SolicitudCompra):
    if solicitud.sucursal not in SUCURSALES:
        raise HTTPException(status_code=422, detail="Sucursal no válida")

    try:
        return registrar_compra(
            solicitud.sucursal,
            [articulo.model_dump() for articulo in solicitud.articulos]
        )
    except ValueError as error:
        raise HTTPException(status_code=400, detail=str(error))
"""

if '@app.post("/compras"' not in codigo:
    ruta.write_text(codigo + nuevo_codigo, encoding="utf-8")
    print("Endpoint POST /compras agregado correctamente.")
else:
    print("El endpoint ya existe. No se modificó el archivo.")

Endpoint POST /compras agregado correctamente.


# Validación del registro de compras

Se realizaron pruebas funcionales del endpoint `POST /compras` utilizando la interfaz interactiva de FastAPI.

**Prueba 1: Compra válida**

Se registró una compra desde la sucursal Cancún con los siguientes artículos:

- SKU001: 1 unidad.
- SKU002: 2 unidades.

La API devolvió el código HTTP 201, asignó el identificador de compra 2 y calculó un importe total de $610.

Posteriormente, mediante `GET /productos`, se verificó que el stock de SKU001 disminuyó de 14 a 13 unidades y el de SKU002 de 38 a 36 unidades.

**Prueba 2: Inventario insuficiente**

Se intentó registrar una compra desde Mérida que incluía:

- SKU001: 1 unidad disponible.
- SKU027: 100 unidades solicitadas, con solamente 8 disponibles.

La API devolvió el código HTTP 400 con el mensaje `Inventario insuficiente para SKU027`.

Se consultó nuevamente el inventario y se comprobó que SKU001 permaneció con 13 unidades y SKU027 con 8.

**Conclusión**

El sistema registra compras válidas y descuenta las existencias del inventario compartido. Cuando algún artículo no tiene suficiente stock, la transacción completa se revierte mediante `rollback`, evitando modificaciones parciales.

Estas pruebas validan el comportamiento esperado en los escenarios evaluados; aún quedan pendientes pruebas adicionales de entradas inválidas y concurrencia.

# API de recomendaciones de productos

Se implementará el endpoint `GET /recomendaciones/{sku}` para consultar productos relacionados con un artículo seleccionado.

**Modelo utilizado**

Las recomendaciones partirán de las probabilidades posteriores calculadas mediante el modelo bayesiano de asociaciones de compra, almacenadas en `Data/asociaciones_modelo.csv`.

Para cada producto de origen, los candidatos se ordenarán de acuerdo con su probabilidad posterior estimada.

**Reglas de negocio**

1. El producto de origen debe existir en el catálogo.
2. No se recomendará el mismo producto que el cliente seleccionó.
3. Solo se mostrarán productos con stock mayor que cero.
4. Las recomendaciones utilizarán el inventario compartido entre las cinco sucursales.
5. La respuesta incluirá información del producto recomendado y la puntuación del modelo.

**Consideración metodológica**

La probabilidad posterior mide la asociación estimada entre productos a partir del historial de compras. No garantiza por sí sola compatibilidad técnica ni causalidad.

Por ello, el sistema deberá distinguir entre asociación estadística y compatibilidad de uso. Se contempla incorporar reglas de negocio para mejorar la pertinencia de las recomendaciones.

**Integración prevista**

El frontend consultará este endpoint cuando el cliente abra su carrito, antes de confirmar la compra, para permitirle agregar productos complementarios.

Crear el endpoint GET /recomendaciones/{sku}

In [71]:
from pathlib import Path

ruta = Path(r"C:\Users\Lucinda\Downloads\Prueba tecnica\backend\main.py")
codigo = ruta.read_text(encoding="utf-8")

nuevo_codigo = '''

import csv
from pathlib import Path

@app.get("/recomendaciones/{sku}")
def obtener_recomendaciones(sku: str, limite: int = 5):

    if not 1 <= limite <= 28:
        raise HTTPException(
            status_code=422,
            detail="El límite debe estar entre 1 y 28"
        )

    with conectar() as conexion:
        origen = conexion.execute(
            "SELECT sku FROM productos WHERE sku = ?",
            (sku,)
        ).fetchone()

        if origen is None:
            raise HTTPException(
                status_code=404,
                detail="Producto no encontrado"
            )

        productos = conexion.execute(
            "SELECT * FROM productos WHERE stock > 0"
        ).fetchall()

    disponibles = {
        producto["sku"]: dict(producto)
        for producto in productos
    }

    ruta_modelo = (
        Path(__file__).resolve().parent.parent
        / "Data"
        / "asociaciones_modelo.csv"
    )

    recomendaciones = []

    with ruta_modelo.open(
        encoding="utf-8-sig",
        newline=""
    ) as archivo:

        lector = csv.DictReader(archivo)

        for fila in lector:
            candidato = fila["sku_recomendado"]

            if fila["sku_origen"] != sku:
                continue

            if candidato == sku or candidato not in disponibles:
                continue

            recomendaciones.append({
                **disponibles[candidato],
                "puntuacion": float(fila["media_posterior"])
            })

    recomendaciones.sort(
        key=lambda producto: producto["puntuacion"],
        reverse=True
    )

    return {
        "sku_origen": sku,
        "recomendaciones": recomendaciones[:limite]
    }
'''

if '@app.get("/recomendaciones/{sku}")' not in codigo:
    ruta.write_text(codigo + nuevo_codigo, encoding="utf-8")
    print("Endpoint de recomendaciones agregado correctamente.")
else:
    print("El endpoint ya existe. No se modificó el archivo.")

Endpoint de recomendaciones agregado correctamente.


# Evaluación  preliminar del sistema de recomendaciones

Se realizó una prueba utilizando el SKU001 (Soplete de gas MAPP). El sistema devolvió cinco productos relacionados y disponibles en inventario.

Los resultados incluyen accesorios, consumibles y equipo de protección asociados con trabajos de soldadura.

La evaluación cualitativa muestra pertinencia temática, aunque la asociación estadística no garantiza compatibilidad técnica. Como mejora futura se propone incorporar reglas de compatibilidad basadas en las características de los productos.

Esta prueba constituye una validación funcional y cualitativa inicial, no una medición formal de precisión predictiva.

In [74]:
from pathlib import Path

ruta = Path("backend/main.py")
codigo = ruta.read_text(encoding="utf-8")

inicio = codigo.find("def obtener_recomendaciones(")
fin = codigo.find("    if not 1 <= limite <= 28:", inicio)

if inicio != -1 and fin != -1:
    original = "def obtener_recomendaciones(sku: str, limite: int = 5):\n\n"
    codigo = codigo[:inicio] + original + codigo[fin:]
    ruta.write_text(codigo, encoding="utf-8")
    print("Endpoint original recuperado correctamente.")
else:
    print("No se modificó el archivo. Necesitamos revisar el código.")

Endpoint original recuperado correctamente.


# 9 Integración del frontend con el backend

####Arquitectura de la aplicación

Se desarrolló una interfaz web utilizando React y Vite, mientras que el backend se implementó con FastAPI en Python.

La aplicación sigue una arquitectura cliente-servidor:

- **Frontend (React):** presenta el catálogo de productos y permite la interacción con el usuario.
- **Backend (FastAPI):** procesa las solicitudes, consulta la base de datos y ejecuta las operaciones del sistema.
- **Base de datos (SQLite):** almacena productos, inventario y transacciones.
- **Modelo de recomendaciones:** utiliza las asociaciones probabilísticas previamente calculadas para sugerir productos relacionados que tengan existencias disponibles.

#### Comunicación mediante API REST

El frontend consulta el endpoint `GET /productos` mediante la función `fetch()` de JavaScript.

El backend devuelve los productos en formato JSON, incluyendo SKU, nombre, categoría, precio y existencias.

Durante la integración se verificó que el endpoint funcionaba correctamente al consultarlo directamente desde el navegador.

####Configuración de CORS

Aunque el backend respondía correctamente, inicialmente el frontend no podía obtener los productos.

Se identificó como posible causa la política de seguridad del navegador conocida como **Cross-Origin Resource Sharing (CORS)**, debido a que React y FastAPI se ejecutan en orígenes diferentes.

Para permitir la comunicación durante el desarrollo, se configurará el middleware `CORSMiddleware` de FastAPI, autorizando los siguientes orígenes:

- `http://localhost:5173`
- `http://127.0.0.1:5173`

Esta configuración se limita al entorno local de desarrollo. En un despliegue de producción deberán autorizarse únicamente los dominios correspondientes.

**Validación pendiente:** comprobar que el frontend puede recuperar y mostrar correctamente los productos después de aplicar la configuración.

In [76]:
from pathlib import Path

ruta = Path("backend/main.py")
codigo = ruta.read_text(encoding="utf-8")

if "from fastapi.middleware.cors import CORSMiddleware" not in codigo:
    codigo = (
        "from fastapi.middleware.cors import CORSMiddleware\n"
        + codigo
    )

bloque = '''
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173", "http://127.0.0.1:5173"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)
'''

if "app.add_middleware(" not in codigo:
    posicion = codigo.find('@app.get("/")')
    if posicion != -1:
        codigo = codigo[:posicion] + bloque + "\n" + codigo[posicion:]
        ruta.write_text(codigo, encoding="utf-8")
        print("CORS configurado correctamente.")
    else:
        print("No se encontró el lugar para insertar CORS.")
else:
    print("Ya existe una configuración de middleware. Revisarla antes de modificar.")

CORS configurado correctamente.


# Validación de la integración
Después de configurar CORS, se verificó que la aplicación React podía consultar correctamente el endpoint GET /productos de FastAPI.

La interfaz mostró los productos almacenados en SQLite, incluyendo sus identificadores, nombres, categorías, precios y existencias actualizadas.

Esta prueba confirma el funcionamiento de la comunicación entre frontend, backend y base de datos.

La integración constituye la primera versión funcional del catálogo web. Las operaciones de carrito, compra y visualización de recomendaciones se incorporarán en las siguientes etapas.

# 10  Documentar el carrito 

El carrito permitirá seleccionar productos del catálogo y especificar sus cantidades.

Antes de confirmar la compra, el sistema mostrará recomendaciones de productos relacionados mediante el modelo probabilístico desarrollado.

Flujo previsto:
1. El cliente selecciona una de las cinco sucursales.
2. Agrega productos al carrito.
3. Consulta su carrito y recibe recomendaciones de productos complementarios disponibles.
4. Puede agregar los productos recomendados antes de finalizar la compra.
5. Al confirmar, el frontend envía la solicitud al endpoint POST /compras.
6. El backend valida las existencias y descuenta las cantidades del inventario compartido mediante una transacción.
Decisión de diseño: las recomendaciones se calculan globalmente, independientemente de la sucursal. La sucursal se registra únicamente como parte de la compra.

Estado: funcionalidad pendiente de implementación y pruebas en la interfaz.

# Estrategia de recomendaciones en el carrito
Para esta primera versión, el sistema genera recomendaciones tomando como referencia únicamente el primer producto agregado al carrito.

El frontend consulta el endpoint GET /recomendaciones/{sku} de FastAPI, solicitando los productos relacionados según las puntuaciones del modelo probabilístico.

Las recomendaciones se presentan antes de confirmar la compra y se excluyen los productos que el cliente ya tiene en el carrito.

Esta estrategia reduce la complejidad de integración y permite validar el funcionamiento del modelo en una experiencia de compra real.

Como mejora futura, se propone generar recomendaciones considerando todos los productos del carrito y combinar sus puntuaciones mediante una estrategia de agregación.

Estado: pendiente de implementación y validación en React.

In [77]:
from backend.database import conectar

with conectar() as conexion:
    ultima_compra = conexion.execute("""
        SELECT
            c.id_compra,
            c.sucursal,
            d.sku,
            d.cantidad,
            d.precio_unitario
        FROM compras c
        JOIN detalle_compras d
            ON c.id_compra = d.id_compra
        WHERE c.id_compra = (
            SELECT MAX(id_compra)
            FROM compras
        )
    """).fetchall()

for fila in ultima_compra:
    print(dict(fila))

{'id_compra': 3, 'sucursal': 'Mérida', 'sku': 'SKU001', 'cantidad': 1, 'precio_unitario': 450}


In [78]:
from backend.database import conectar

with conectar() as conexion:
    producto = conexion.execute("""
        SELECT sku, nombre, stock
        FROM productos
        WHERE sku = 'SKU001'
    """).fetchone()

print(dict(producto))

{'sku': 'SKU001', 'nombre': 'Soplete de gas MAPP', 'stock': 12}


In [79]:
import os
print(os.getcwd())

C:\Users\Lucinda\Downloads\Prueba tecnica


In [80]:
from pathlib import Path

carpetas = [
    Path.home() / "Documents",
    Path.home() / "Downloads" / "Prueba tecnica",
    Path.home() / "Desktop"
]

for carpeta in carpetas:
    if carpeta.exists():
        for archivo in carpeta.rglob("01_exploracion_datos.ipynb"):
            print(archivo)

C:\Users\Lucinda\Documents\01_exploracion_datos.ipynb
C:\Users\Lucinda\Downloads\Prueba tecnica\notebooks\01_exploracion_datos.ipynb
